# Model: from MODIS to VIIRS
**Goal:** from what MODIS sees in a piece of the map during a month, estimate what VIIRS would have seen in the same place and month. The model learns in the years when both sensors operated together and is then applied to 2002–2011, when only MODIS existed. The result is a long series in "VIIRS units".

```
Inputs: MODIS in the cell and month               Outputs: VIIRS in the same cell and month
  Terra and Aqua detections and FRP        →        detections
  fire in neighbouring cells and months    →        total FRP
  position of the Sun, place and date
```

**Steps**
1. Setup
2. Cell × month table (SQL)
3. Calculated variables (feature engineering)
4. Splitting years
5. Three models: baseline, LightGBM and neural network
6. How well do they predict?
7. Which inputs does the model use?
8. What about MODIS's drifting orbits?
9. Year-by-year validation
10. Where does it go wrong? (map)
11. Harmonized series 2002–2025, with its uncertainty range
12. Before and after harmonizing

The first time, running everything takes ~25 min. The table and the validation are saved in `data/modelo/`, and after that everything takes ~10 min.

---
## 1. Setup
The charts live in [graficas_modelo.py](graficas_modelo.py) and the style in [estilo.py](estilo.py), so only the model logic stays here.

In [ ]:
import sys
from pathlib import Path

import duckdb
import lightgbm as lgb
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import r2_score
from torch import nn

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "notebooks"))
import graficas_modelo as graf

# LightGBM and PyTorch each ship their own OpenMP, and on a Mac they block each other.
# With a single thread PyTorch does not use OpenMP, and the network still trains an epoch in ~5 s.
torch.set_num_threads(1)
_ = torch.manual_seed(42)

GRADO = 0.5  # size of each map cell, in degrees (~55 km)
ARCHIVO_TABLA = ROOT / "data" / "modelo" / f"tabla_celdas_{GRADO}.parquet"

# In-memory database for the intermediate steps; the NASA database is opened read-only
con = duckdb.connect()
con.execute(f"ATTACH '{ROOT / 'data' / 'db_nasa.duckdb'}' AS nasa (READ_ONLY)")

---
## 2. Cell × month table
Built in SQL in 7 steps. Each step is a **view**: a named query that computes nothing until it is used. Any step can be checked with `con.sql("SELECT * FROM view_name LIMIT 5")`.

**Step 1 · Each detection with its cell and local time.** The cell comes from dividing latitude and longitude by 0.5°. Local solar time is UTC time plus longitude ÷ 15, because every 15° of longitude is one hour.

In [ ]:
con.execute(f"""
CREATE OR REPLACE VIEW detecciones AS
SELECT
    satellite, sensor, daynight, type, frp, acq_date, latitude,
    CAST(floor(latitude / {GRADO}) AS INT)  AS fila,
    CAST(floor(longitude / {GRADO}) AS INT) AS col,
    year(acq_date)                          AS año,
    month(acq_date)                         AS mes,
    scan * track                            AS area_px,     -- pixel area, km²
    TRY_CAST(confidence AS INT)             AS confianza,   -- 0–100 in MODIS (VIIRS uses letters)
    ((acq_time // 100) + (acq_time % 100) / 60 + longitude / 15 + 48) % 24 AS hora_local
FROM nasa.hotspots
""")

**Step 2 · Good months for each satellite.** A month counts if the satellite has data every day and none with a failure. A day has a failure if the satellite detected less than 25 % of its normal, measured with the median of the 45 days before and after. Example: Aqua from 16 August to 2 September 2020. It happens on 37 of 22,690 days.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW meses AS
WITH por_dia AS (
    SELECT satellite, año, mes, acq_date, count(*) AS n
    FROM detecciones
    GROUP BY ALL
),
con_falla AS (
    SELECT *,
           n < 0.25 * median(n) OVER (PARTITION BY satellite ORDER BY acq_date
                                      RANGE BETWEEN INTERVAL 45 DAYS PRECEDING
                                                AND INTERVAL 45 DAYS FOLLOWING) AS falla
    FROM por_dia
),
por_satelite AS (
    SELECT satellite, año, mes,
           count(*) = day(last_day(make_date(año, mes, 1))) AND NOT bool_or(falla) AS bueno
    FROM con_falla
    GROUP BY satellite, año, mes
)
SELECT año, mes,
       bool_or(satellite = 'Terra' AND bueno) AS terra_ok,
       bool_or(satellite = 'Aqua'  AND bueno) AS aqua_ok,
       CASE WHEN bool_or(satellite = 'N'   AND bueno) THEN 'N'
            WHEN bool_or(satellite = 'N20' AND bueno) THEN 'N20' END AS viirs_sat   -- the active VIIRS
FROM por_satelite
GROUP BY año, mes
""")

**Step 3 · Terra and Aqua overpass time each month.** It is the median local time of their daytime detections near the equator. It shows how the orbits drift from 2022 on.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW paso AS
SELECT año, mes,
       median(hora_local) FILTER (satellite = 'Terra') AS hora_paso_terra,
       median(hora_local) FILTER (satellite = 'Aqua')  AS hora_paso_aqua
FROM detecciones
WHERE sensor = 'MODIS' AND daynight = 'D' AND abs(latitude) < 30
GROUP BY año, mes
""")

**Step 4 · MODIS per cell and month (the inputs).**

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW modis AS
SELECT fila, col, año, mes,
       count(*) FILTER (satellite = 'Terra' AND daynight = 'D') AS terra_dia,
       count(*) FILTER (satellite = 'Terra' AND daynight = 'N') AS terra_noche,
       count(*) FILTER (satellite = 'Aqua'  AND daynight = 'D') AS aqua_dia,
       count(*) FILTER (satellite = 'Aqua'  AND daynight = 'N') AS aqua_noche,
       coalesce(sum(frp) FILTER (satellite = 'Terra'), 0)       AS terra_frp,
       coalesce(sum(frp) FILTER (satellite = 'Aqua'), 0)        AS aqua_frp,
       max(frp)                                 AS modis_frp_max,
       avg(area_px)                             AS modis_area_px,         -- grows toward the edge of the swath
       avg(confianza)                           AS modis_confianza,
       avg(CAST(confianza < 30 AS INT))         AS modis_frac_baja_conf,
       count(DISTINCT acq_date)                 AS modis_dias,            -- days with fire in the month
       avg(CAST(type = 2 AS INT))               AS modis_frac_fijas       -- industry and other static sources
FROM detecciones
WHERE sensor = 'MODIS'
GROUP BY fila, col, año, mes
""")

**Step 5 · Fire in the 8 neighbouring cells.** Each cell passes its count to its 8 neighbours, and then each cell adds up what it received.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW vecinos AS
SELECT m.fila + df AS fila,
       m.col + dc  AS col,
       m.año, m.mes,
       sum(m.terra_dia + m.terra_noche + m.aqua_dia + m.aqua_noche) AS modis_vecinos
FROM modis m
CROSS JOIN (SELECT unnest([-1, 0, 1]) AS df)
CROSS JOIN (SELECT unnest([-1, 0, 1]) AS dc)
WHERE NOT (df = 0 AND dc = 0)   -- not counting the cell itself
GROUP BY ALL
""")

**Step 6 · VIIRS per cell and month (what we want to predict).**

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW viirs AS
SELECT fila, col, año, mes, satellite,
       count(*) AS viirs_n,
       sum(frp) AS viirs_frp
FROM detecciones
WHERE sensor LIKE 'VIIRS%'
GROUP BY fila, col, año, mes, satellite
""")

**Step 7 · Join everything.** There is one row for every cell that ever had fire and for every month. Months without detections are included, because otherwise the model would never learn to predict zero.

Three more columns are added: the fire of the previous month, of the next month, and the historical MODIS average in the cell for that month of the year.

The result is saved to a file to avoid recomputing: the first time takes ~1 min. **If you change anything in the SQL, delete that file** so it is rebuilt.

In [ ]:
con.execute(f"""
CREATE OR REPLACE VIEW tabla_celdas AS
WITH celdas AS (SELECT DISTINCT fila, col FROM detecciones),
rejilla AS (   -- every cell × every month, with zeros where there were no detections
    SELECT c.fila, c.col, m.*, p.hora_paso_terra, p.hora_paso_aqua,
           coalesce(md.terra_dia, 0)            AS terra_dia,
           coalesce(md.terra_noche, 0)          AS terra_noche,
           coalesce(md.aqua_dia, 0)             AS aqua_dia,
           coalesce(md.aqua_noche, 0)           AS aqua_noche,
           coalesce(md.terra_dia + md.terra_noche + md.aqua_dia + md.aqua_noche, 0) AS modis_n,
           coalesce(md.terra_frp, 0)            AS terra_frp,
           coalesce(md.aqua_frp, 0)             AS aqua_frp,
           coalesce(md.modis_frp_max, 0)        AS modis_frp_max,
           md.modis_area_px,                    -- no detections means no pixel or confidence: left empty
           md.modis_confianza,
           coalesce(md.modis_frac_baja_conf, 0) AS modis_frac_baja_conf,
           coalesce(md.modis_dias, 0)           AS modis_dias,
           coalesce(md.modis_frac_fijas, 0)     AS modis_frac_fijas,
           coalesce(vc.modis_vecinos, 0)        AS modis_vecinos,
           -- VIIRS only in its good months; in the others it is left empty (unknown)
           CASE WHEN m.viirs_sat IS NOT NULL THEN coalesce(v.viirs_n, 0) END   AS viirs_n,
           CASE WHEN m.viirs_sat IS NOT NULL THEN coalesce(v.viirs_frp, 0) END AS viirs_frp
    FROM celdas c
    CROSS JOIN meses m
    LEFT JOIN paso p     ON (p.año, p.mes) = (m.año, m.mes)
    LEFT JOIN modis md   ON (md.fila, md.col, md.año, md.mes) = (c.fila, c.col, m.año, m.mes)
    LEFT JOIN vecinos vc ON (vc.fila, vc.col, vc.año, vc.mes) = (c.fila, c.col, m.año, m.mes)
    LEFT JOIN viirs v    ON (v.fila, v.col, v.año, v.mes, v.satellite) = (c.fila, c.col, m.año, m.mes, m.viirs_sat)
)
SELECT *,
       ({GRADO} * (fila + 0.5))::FLOAT AS lat,   -- cell centre
       ({GRADO} * (col + 0.5))::FLOAT  AS lon,
       lag(modis_n)  OVER celda AS modis_mes_anterior,
       lead(modis_n) OVER celda AS modis_mes_siguiente,
       avg(CASE WHEN año BETWEEN 2003 AND 2021 AND terra_ok AND aqua_ok THEN modis_n END)
           OVER (PARTITION BY fila, col, mes) AS modis_historico   -- years with normal orbits
FROM rejilla
WINDOW celda AS (PARTITION BY fila, col ORDER BY año, mes)
""")

if not ARCHIVO_TABLA.exists():
    ARCHIVO_TABLA.parent.mkdir(exist_ok=True)
    con.execute(f"COPY (SELECT * FROM tabla_celdas ORDER BY año, mes, fila, col) TO '{ARCHIVO_TABLA}'")

tabla = pd.read_parquet(ARCHIVO_TABLA)
numericas = tabla.columns.difference(["fila", "col", "año", "mes", "terra_ok", "aqua_ok", "viirs_sat"])
tabla[numericas] = tabla[numericas].astype("float32")  # half the memory
print(f"{len(tabla):,} rows = {tabla[['fila', 'col']].drop_duplicates().shape[0]:,} cells "
      f"× {tabla[['año', 'mes']].drop_duplicates().shape[0]} months")

---
## 3. Calculated variables (feature engineering)
New columns computed from the existing ones:

| Variable | What it is | Why it could help |
|---|---|---|
| `horas_de_sol` | day length in the cell | fire has a daily cycle |
| `cenit_terra`, `cenit_aqua` | angle of the Sun from the vertical when the satellite passes (0° = Sun overhead) | with a high Sun the ground is hotter and a fire is harder to tell apart |
| `distancia_sol` | Earth–Sun distance (0.983 in January, 1.017 in July) | we were asked to try it; it only depends on the date |
| `frp_por_det_terra`, `frp_por_det_aqua` | average FRP per detection | small fires are the ones VIIRS sees and MODIS misses |
| `razon_aqua_terra` | log(Aqua ÷ Terra) | if it is high, fire concentrates in the afternoon, when VIIRS passes |
| `frac_noche` | % of MODIS detections at night | at night VIIRS sees ~26 times more than Aqua; by day, ~6 times |

**Why the Sun angle uses the nominal time and not the real one:** we first tried the real overpass time of each month (step 3). That time barely changes in the training years, but from 2022 it leaves the range: Aqua ends up passing at 14:46. The neural network never saw those times, extrapolated without control and missed by **58 %** in 2022–2025. With the nominal time (10:30 for Terra and 13:30 for Aqua), the angle only depends on place and month, and the model stays stable.

In [ ]:
DIA_DEL_AÑO = np.array([15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349])  # the 15th of each month


HORA_NOMINAL = {"terra": 10.5, "aqua": 13.5}  # design local solar time of each orbit


def variables_sol(t):
    """Position of the Sun in mid-month (standard astronomical formulas)."""
    dia = DIA_DEL_AÑO[t.mes - 1]
    lat = np.radians(t.lat)
    declinacion = np.radians(23.44) * np.sin(2 * np.pi * (284 + dia) / 365)  # latitude where the Sun is overhead
    t["distancia_sol"] = 1 - 0.01672 * np.cos(2 * np.pi * (dia - 4) / 365.25)
    t["horas_de_sol"] = 24 / np.pi * np.arccos(np.clip(-np.tan(lat) * np.tan(declinacion), -1, 1))
    for sat in ("terra", "aqua"):
        angulo_horario = np.radians(15 * (HORA_NOMINAL[sat] - 12))  # 15° per hour from noon
        cos_cenit = np.sin(lat) * np.sin(declinacion) + np.cos(lat) * np.cos(declinacion) * np.cos(angulo_horario)
        t[f"cenit_{sat}"] = np.degrees(np.arccos(np.clip(cos_cenit, -1, 1)))
    return t


def variables_intensidad(t):
    """How large the fires are and at what time of day they burn."""
    terra = t.terra_dia + t.terra_noche
    aqua = t.aqua_dia + t.aqua_noche
    t["frp_por_det_terra"] = (t.terra_frp / terra).where(terra > 0, 0)
    t["frp_por_det_aqua"] = (t.aqua_frp / aqua).where(aqua > 0, 0)
    t["razon_aqua_terra"] = np.log1p(aqua) - np.log1p(terra)
    t["frac_noche"] = ((t.terra_noche + t.aqua_noche) / t.modis_n).where(t.modis_n > 0, 0)
    return t


tabla = variables_intensidad(variables_sol(tabla))
nuevas = ["horas_de_sol", "cenit_terra", "cenit_aqua", "distancia_sol",
          "frp_por_det_terra", "frp_por_det_aqua", "razon_aqua_terra", "frac_noche"]
tabla[nuevas] = tabla[nuevas].astype("float32")
tabla[nuevas].describe().round(2)

All the model inputs, grouped:

In [ ]:
GRUPOS = {
    "MODIS": ["terra_dia", "terra_noche", "aqua_dia", "aqua_noche", "terra_frp", "aqua_frp",
              "modis_frp_max", "modis_area_px", "modis_confianza", "modis_frac_baja_conf",
              "modis_dias", "modis_frac_fijas"],
    "intensity": ["frp_por_det_terra", "frp_por_det_aqua", "razon_aqua_terra", "frac_noche"],
    "neighbourhood and time": ["modis_vecinos", "modis_mes_anterior", "modis_mes_siguiente", "modis_historico"],
    "sun": ["cenit_terra", "cenit_aqua", "horas_de_sol", "distancia_sol"],
    "place and date": ["lat", "lon", "mes"],
}
ENTRADAS = [entrada for grupo in GRUPOS.values() for entrada in grupo]
print(len(ENTRADAS), "inputs")

---
## 4. Splitting years
Whole years are held out, not random rows. Neighbouring months are very similar, and mixing them would make the model look better than it is.

| Set | Years | Purpose |
|---|---|---|
| training | 2012–2016, 2018–2020 | learning (normal MODIS orbits) |
| test | 2017 (S-NPP), 2021 (NOAA-20) | measuring how well it predicts years it did not see |
| drifting orbits | 2022–2025 | checking whether the model survives the drift (not used for learning) |
| MODIS only | 2002–2011 | applying the model |

In [ ]:
AÑOS_ENTRENAMIENTO = [2012, 2013, 2014, 2015, 2016, 2018, 2019, 2020]
AÑOS_PRUEBA = [2017, 2021]

modis_bueno = tabla.terra_ok & tabla.aqua_ok
con_viirs = modis_bueno & tabla.viirs_sat.notna()

entrenamiento = tabla[con_viirs & tabla.año.isin(AÑOS_ENTRENAMIENTO)]
prueba = tabla[con_viirs & tabla.año.isin(AÑOS_PRUEBA)]
todo_viirs = tabla[con_viirs]  # 2012–2025: to see the drift
solo_modis = tabla[modis_bueno]  # 2002–2025: for the harmonized series

print(f"training: {len(entrenamiento):,} rows · test: {len(prueba):,} rows")

---
## 5. Three models
All three are used the same way:
- `modelo.entrenar(datos)` trains.
- `modelo.predecir(datos)` returns a table with `detecciones` (detections) and `frp` for each row.

With `predecir(datos, satelite="N20")` you choose which VIIRS to imitate: `"N"` for S-NPP or `"N20"` for NOAA-20. If it is not given, it uses the one that was active that month.

### 5.1 Baseline
The simplest approach: multiply MODIS by the average VIIRS/MODIS ratio of its region and month, as seen in the EDA. Any model has to beat this one. Its weak point is that, if MODIS saw nothing in the cell, it predicts zero even when VIIRS did see small fires.

In [ ]:
def region_10_grados(datos):
    """10° × 10° block that each cell belongs to."""
    return ((datos.lat + 90) // 10 * 36 + (datos.lon + 180) // 10).astype(int)


def razones(sumas):
    """VIIRS / MODIS ratio for detections and FRP (empty if MODIS saw nothing)."""
    return pd.DataFrame({
        "detecciones": sumas.viirs_n / sumas.modis_n,
        "frp": sumas.viirs_frp / sumas.modis_frp,
    }).replace(np.inf, np.nan)


class Referencia:
    """VIIRS = MODIS × VIIRS/MODIS ratio of its region (10° × 10°) in that month of the year.
    If the region had no fire in training, it uses the whole world's ratio for that month."""

    def entrenar(self, datos):
        sumas = datos[["mes", "modis_n", "viirs_n", "viirs_frp"]].assign(
            region=region_10_grados(datos), modis_frp=datos.terra_frp + datos.aqua_frp)
        self.razon_region = razones(sumas.groupby(["region", "mes"]).sum())
        self.razon_mes = razones(sumas.drop(columns="region").groupby("mes").sum())
        return self

    def predecir(self, datos, satelite=None):  # does not tell S-NPP and NOAA-20 apart
        llave = pd.MultiIndex.from_arrays([region_10_grados(datos), datos.mes])
        razon = self.razon_region.reindex(llave).to_numpy()
        respaldo = self.razon_mes.reindex(datos.mes).to_numpy()
        razon = np.where(np.isnan(razon), respaldo, razon)
        modis = np.column_stack([datos.modis_n, datos.terra_frp + datos.aqua_frp])
        return pd.DataFrame(modis * razon, columns=["detecciones", "frp"], index=datos.index)


referencia = Referencia().entrenar(entrenamiento)

### 5.2 LightGBM
Decision trees in sequence, where each tree corrects the error of the previous ones. It is usually the strongest method for tabular data.

Two models are trained:
- **Detections,** with a *Poisson* loss, made for counts.
- **FRP,** with a *Tweedie* loss, for values with many zeros.

Takes ~3 min.

In [ ]:
class ModeloLightGBM:
    def __init__(self, arboles=600, paso=0.05):
        """More trees with shorter steps = more accurate but slower."""
        self.parametros = dict(n_estimators=arboles, learning_rate=paso, num_leaves=127, min_child_samples=100,
                               subsample=0.5, subsample_freq=1, colsample_bytree=0.9, verbose=-1)

    def entradas(self, datos, satelite=None):
        """The inputs + a column saying which VIIRS to imitate (0 = S-NPP, 1 = NOAA-20)."""
        satelite = datos.viirs_sat if satelite is None else satelite
        return datos[ENTRADAS].assign(noaa20=np.where(satelite == "N20", 1, 0))

    def entrenar(self, datos):
        x = self.entradas(datos)
        self.detecciones = lgb.LGBMRegressor(objective="poisson", **self.parametros).fit(x, datos.viirs_n)
        self.frp = lgb.LGBMRegressor(objective="tweedie", **self.parametros).fit(x, datos.viirs_frp)
        return self

    def predecir(self, datos, satelite=None):
        x = self.entradas(datos, satelite)
        return pd.DataFrame({"detecciones": self.detecciones.predict(x), "frp": self.frp.predict(x)},
                            index=datos.index)

    def importancia(self):
        """% of the trees' total gain contributed by each input."""
        tabla = pd.DataFrame({"detections": self.detecciones.booster_.feature_importance("gain"),
                              "FRP": self.frp.booster_.feature_importance("gain")},
                             index=self.detecciones.feature_name_)
        return (100 * tabla / tabla.sum()).round(1)


lightgbm = ModeloLightGBM().entrenar(entrenamiento)

### 5.3 Neural network
The same variables go in and 4 values come out: S-NPP detections and FRP, and NOAA-20 detections and FRP.

```
27 inputs → 256 → 128 → 64 neurons → [det S-NPP, det NOAA-20, FRP S-NPP, FRP NOAA-20]
```

In this data S-NPP and NOAA-20 never overlap in time. So in each row only the 2 outputs of the satellite active that month are corrected. Takes ~1 min.

In [ ]:
def error_poisson(log_prediccion, real, usar):
    """Average Poisson error (deviance / 2), only on the observed outputs. 0 = perfect."""
    error = torch.exp(log_prediccion) - real * log_prediccion + torch.special.xlogy(real, real) - real
    return (error * usar).sum() / usar.sum()


class RedNeuronal:
    SALIDAS = ["det S-NPP", "det NOAA-20", "FRP S-NPP", "FRP NOAA-20"]
    ESCALA_FRP = 100  # FRP is learned in hundreds of MW, to be on a scale similar to the counts
    EN_LOG = ["terra_dia", "terra_noche", "aqua_dia", "aqua_noche", "terra_frp", "aqua_frp",
              "modis_frp_max", "modis_dias", "frp_por_det_terra", "frp_por_det_aqua",
              "modis_vecinos", "modis_mes_anterior", "modis_mes_siguiente", "modis_historico"]

    def preparar(self, datos, ajustar=False):
        """Inputs ready for the network:
        - counts and FRP in log, so the numbers have a similar size
        - month and longitude as angles (sine and cosine), so December sits next to January
        - everything standardized (mean 0, deviation 1) with the training mean and deviation"""
        x = datos[ENTRADAS].fillna(0)
        x = x.assign(**{c: np.log1p(x[c].clip(lower=0)) for c in self.EN_LOG},
                     mes_sen=np.sin(2 * np.pi * x.mes / 12), mes_cos=np.cos(2 * np.pi * x.mes / 12),
                     lon_sen=np.sin(np.radians(x.lon)), lon_cos=np.cos(np.radians(x.lon)))
        x = x.drop(columns=["mes", "lon"]).to_numpy("float32")
        if ajustar:
            self.media, self.desviacion = x.mean(axis=0), x.std(axis=0) + 1e-6
        return torch.tensor((x - self.media) / self.desviacion)

    def objetivos(self, datos):
        """What the network must learn, and which outputs were observed (those of the active VIIRS)."""
        es_noaa20 = datos.viirs_sat == "N20"
        y = pd.DataFrame({
            "det S-NPP": datos.viirs_n.where(~es_noaa20),
            "det NOAA-20": datos.viirs_n.where(es_noaa20),
            "FRP S-NPP": datos.viirs_frp.where(~es_noaa20) / self.ESCALA_FRP,
            "FRP NOAA-20": datos.viirs_frp.where(es_noaa20) / self.ESCALA_FRP,
        })
        return torch.tensor(y.fillna(0).to_numpy("float32")), torch.tensor(y.notna().to_numpy("float32"))

    def entrenar(self, datos, epocas=10, lote=8192):
        x = self.preparar(datos, ajustar=True)
        y, usar = self.objetivos(datos)
        self.red = nn.Sequential(
            nn.Linear(x.shape[1], 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 4),  # outputs are in log, so the prediction is always positive
        )
        with torch.no_grad():  # starts by predicting the average of each output
            self.red[-1].bias.copy_(torch.log((y * usar).sum(axis=0) / usar.sum(axis=0)))
        optimizador = torch.optim.Adam(self.red.parameters(), lr=1e-3)
        ritmo = torch.optim.lr_scheduler.CosineAnnealingLR(optimizador, T_max=epocas)  # steps get shorter and shorter

        for epoca in range(1, epocas + 1):
            error_total = 0.0
            for filas in torch.randperm(len(x)).split(lote):  # random batches
                error = error_poisson(self.red(x[filas]), y[filas], usar[filas])
                optimizador.zero_grad()
                error.backward()
                nn.utils.clip_grad_norm_(self.red.parameters(), 1.0)
                optimizador.step()
                error_total += error.item() * len(filas)
            ritmo.step()
            print(f"epoch {epoca:2d}/{epocas} · error {error_total / len(x):.2f}")
        return self

    def predecir(self, datos, satelite=None, bloque=1_000_000):
        with torch.no_grad():  # in chunks, so memory does not fill up
            partes = [torch.exp(self.red(self.preparar(datos.iloc[i:i + bloque]))).numpy()
                      for i in range(0, len(datos), bloque)]
        salidas = pd.DataFrame(np.vstack(partes), columns=self.SALIDAS, index=datos.index)
        satelite = datos.viirs_sat if satelite is None else satelite
        es_noaa20 = np.asarray(satelite == "N20")
        return pd.DataFrame({
            "detecciones": np.where(es_noaa20, salidas["det NOAA-20"], salidas["det S-NPP"]),
            "frp": np.where(es_noaa20, salidas["FRP NOAA-20"], salidas["FRP S-NPP"]) * self.ESCALA_FRP,
        }, index=datos.index)


red = RedNeuronal().entrenar(entrenamiento)

---
## 6. How well do they predict?
Measured on the test years, which no model saw during training:
- **Error in the total:** how much it overshoots (+) or falls short (−) when adding up the whole world and all months. 0 % is perfect.
- **Typical monthly error:** in any given month, how far off it is in the global total. It is the median over months.
- **R² per cell:** how well it gets each cell right, on a log scale. 1 is perfect; 0 means no better than guessing the average.

In [ ]:
def totales_mensuales(datos, predicciones):
    """Whole-world sum for each month: the observed value and each model's prediction."""
    columnas = {("detecciones", "Observed"): datos.viirs_n, ("frp", "Observed"): datos.viirs_frp}
    for nombre, prediccion in predicciones.items():
        columnas[("detecciones", nombre)] = prediccion.detecciones
        columnas[("frp", nombre)] = prediccion.frp
    return pd.DataFrame(columnas).groupby([datos.año, datos.mes]).sum()


def evaluar(datos, predicciones):
    totales = totales_mensuales(datos, predicciones)
    filas = []
    for variable, observado in (("detecciones", datos.viirs_n), ("frp", datos.viirs_frp)):
        for nombre, prediccion in predicciones.items():
            mensual = totales[(variable, nombre)] / totales[(variable, "Observed")] - 1
            filas.append({
                "variable": variable,
                "model": nombre,
                "error in the total (%)": 100 * (prediccion[variable].sum() / observado.sum() - 1),
                "typical monthly error (%)": 100 * mensual.abs().median(),
                "R² per cell": r2_score(np.log1p(observado), np.log1p(prediccion[variable].clip(lower=0))),
            })
    return pd.DataFrame(filas).set_index(["variable", "model"]).round(3)

In [ ]:
modelos = {"Baseline": referencia, "LightGBM": lightgbm, "Neural network": red}
predicciones = {nombre: modelo.predecir(prueba) for nombre, modelo in modelos.items()}

display(evaluar(prueba, predicciones))
graf.prueba(totales_mensuales(prueba, predicciones), satelites={2017: "VIIRS S-NPP", 2021: "VIIRS NOAA-20"})

---
## 7. Which inputs does the model use?
First, each group of new variables was tested separately: one group was added at a time and the result measured with a fast version of LightGBM (300 trees). These were the results on the test years:

| Inputs | R² detections | Monthly error detections | R² FRP | Monthly error FRP |
|---|---|---|---|---|
| the 15 original ones (MODIS, place and date) | 0.812 | 1.62 % | 0.763 | 1.75 % |
| + Sun and real overpass time | 0.813 | 1.69 % | 0.763 | 1.42 % |
| + intensity | 0.812 | 1.63 % | 0.764 | 1.56 % |
| + neighbourhood | 0.818 | 1.48 % | 0.775 | 1.48 % |
| + previous and next month | 0.819 | 1.49 % | 0.774 | 1.79 % |
| **all** | **0.824** | **1.29 %** | **0.783** | **1.18 %** |

No group helps much on its own, but together they cut the monthly error by 20 % to 30 %. The chart shows how much the final model uses each input.

In [ ]:
graf.importancia(lightgbm.importancia(), GRUPOS)

---
## 8. What about MODIS's drifting orbits?
Since 2022, Terra passes earlier and earlier and Aqua later and later. In 2025, Terra passes at 9:25 instead of 10:30, and Aqua at 14:46 instead of 13:30.

The years 2022–2025 were not used for training, and the model does not know when each satellite really passed. If the error grows in those years, the model is only reliable with normal orbits, like those of 2002–2011, which is where it is applied.

In [ ]:
def a_fecha(indice):
    """(year, month) index → date of the first day of the month."""
    return pd.to_datetime([f"{año}-{mes:02d}-01" for año, mes in indice])


def errores_mensuales(datos, modelos):
    """Error (%) of the global total of each month, for each model. + = overpredicts."""
    totales = totales_mensuales(datos, {nombre: modelo.predecir(datos) for nombre, modelo in modelos.items()})
    errores = pd.DataFrame({
        (variable, nombre): 100 * (totales[(variable, nombre)] / totales[(variable, "Observed")] - 1)
        for variable in ("detecciones", "frp") for nombre in modelos
    })
    errores.index = a_fecha(errores.index)
    return errores


errores = errores_mensuales(todo_viirs, modelos)
graf.deriva(errores, AÑOS_PRUEBA)

periodo = np.select([errores.index.year >= 2022, errores.index.year.isin(AÑOS_PRUEBA)],
                    ["drifting orbits (2022–2025)", "test years"], "training years")
errores.abs().groupby(periodo).median().round(1).rename_axis("typical monthly error (%)")

---
## 9. Year-by-year validation
So far the error was measured on only 2 test years. To measure it on the 10 years of 2012–2021, LightGBM is trained 10 times, leaving out a different year each time, and the left-out year is predicted. That way every month of 2012–2021 has a prediction made without having seen it.

This validation gives two things:
- **Where it goes wrong:** the map in section 10.
- **How far off it can be in a month:** the uncertainty range in section 11.

A fast version of LightGBM is used (300 trees with longer steps), which gave almost the same accuracy in the experiment of section 7. The first time takes ~12 min; the result is saved in `data/modelo/`.

In [ ]:
ARCHIVO_VALIDACION = ROOT / "data" / "modelo" / "validacion_por_año.parquet"
AÑOS_VALIDACION = range(2012, 2022)

if not ARCHIVO_VALIDACION.exists():
    datos = tabla[con_viirs & tabla.año.isin(AÑOS_VALIDACION)]
    partes = []
    for año in AÑOS_VALIDACION:
        modelo = ModeloLightGBM(arboles=300, paso=0.1).entrenar(datos[datos.año != año])
        fuera = datos[datos.año == año]
        partes.append(fuera[["lat", "lon", "año", "mes", "viirs_n", "viirs_frp"]].join(modelo.predecir(fuera)))
        print(año, "done")
    pd.concat(partes).to_parquet(ARCHIVO_VALIDACION)

validacion = pd.read_parquet(ARCHIVO_VALIDACION)
evaluar(validacion, {"LightGBM (one year left out at a time)": validacion[["detecciones", "frp"]]})

---
## 10. Where does it go wrong?
Cells are grouped into 2° blocks (~220 km) and, with the validation predictions, we measure:
- **Bias:** whether, over the 2012–2021 total, the model overshoots (red) or falls short (blue) in that block.
- **Typical error of a month:** how far off it is in any given month in that block. Only months with at least 10 detections count, because with few detections percentages mislead.

Blocks with fewer than 1,000 VIIRS detections in 10 years are grey.

In [ ]:
def error_por_bloque(validacion, grado=2, minimo=1000):
    v = validacion.assign(bloque_lat=validacion.lat // grado * grado, bloque_lon=validacion.lon // grado * grado)
    por_mes = v.groupby(["bloque_lat", "bloque_lon", "año", "mes"])[["viirs_n", "detecciones"]].sum()
    por_mes["error"] = 100 * (por_mes.detecciones / por_mes.viirs_n - 1)

    bloques = por_mes.groupby(level=["bloque_lat", "bloque_lon"]).agg(
        observado=("viirs_n", "sum"), estimado=("detecciones", "sum"))
    bloques["sesgo"] = 100 * (bloques.estimado / bloques.observado - 1)
    bloques["error_mensual"] = por_mes[por_mes.viirs_n >= 10].error.abs().groupby(
        level=["bloque_lat", "bloque_lon"]).median()
    bloques["confiable"] = bloques.observado >= minimo
    return bloques.reset_index()


bloques = error_por_bloque(validacion)
graf.mapa_error(bloques, grado=2)

# What share of the world's fire is in blocks where the model does well?
medidos = bloques[bloques.confiable]
for limite in (10, 20):
    parte = medidos.observado[medidos.sesgo.abs() <= limite].sum() / bloques.observado.sum()
    print(f"{100 * parte:.0f} % of VIIRS detections are in blocks with a bias below ±{limite} %")

---
## 11. Harmonized series 2002–2025
This is the final result: LightGBM applied to every cell and every month in which MODIS has complete data, in VIIRS NOAA-20 units.
- **2012–2025:** can be compared with what VIIRS really observed.
- **Before 2012:** it is what VIIRS would have seen.

**Uncertainty range:** it comes from the validation. It is the range in which the error of the world total fell in 90 % of the months of 2012–2021. For a small region the range is wider; the previous map shows where.

Two files are saved in `data/modelo/`:
- `armonizado_celdas.parquet`: every cell and month. It is the basis of the burn calendar per area.
- `serie_armonizada.csv`: the world total of each month.

In [ ]:
error_mensual = validacion.groupby(["año", "mes"])[["viirs_n", "viirs_frp", "detecciones", "frp"]].sum()
RANGO = {
    "detecciones": (error_mensual.detecciones / error_mensual.viirs_n - 1).quantile([0.05, 0.95]).to_numpy(),
    "frp": (error_mensual.frp / error_mensual.viirs_frp - 1).quantile([0.05, 0.95]).to_numpy(),
}
for variable, (bajo, alto) in RANGO.items():
    print(f"{variable}: in 90 % of months the error of the world total was between {100 * bajo:+.1f} % and {100 * alto:+.1f} %")

In [ ]:
# Every cell and month, in VIIRS NOAA-20 units (~2 min)
celdas_armonizadas = solo_modis[["fila", "col", "lat", "lon", "año", "mes"]].join(
    lightgbm.predecir(solo_modis, satelite="N20"))
celdas_armonizadas.to_parquet(ROOT / "data" / "modelo" / "armonizado_celdas.parquet")


def serie_armonizada(celdas):
    """World total for each month: MODIS as is, harmonized MODIS (with its 90 % range) and observed VIIRS."""
    modis = solo_modis.assign(frp=solo_modis.terra_frp + solo_modis.aqua_frp).groupby(["año", "mes"]).agg(
        detecciones=("modis_n", "sum"), frp=("frp", "sum"))
    estimado = celdas.groupby(["año", "mes"])[["detecciones", "frp"]].sum()
    observado = tabla[tabla.viirs_sat.notna()].groupby(["año", "mes"]).agg(
        detecciones=("viirs_n", "sum"), frp=("viirs_frp", "sum"), satelite=("viirs_sat", "first"))
    serie = pd.concat([modis.add_suffix("_modis"), estimado.add_suffix("_estimado"),
                       observado.add_suffix("_observado")], axis=1).sort_index()
    for variable, (bajo, alto) in RANGO.items():  # if the model overshoots by X %, the real value is estimate / (1 + X)
        serie[f"{variable}_min"] = serie[f"{variable}_estimado"] / (1 + alto)
        serie[f"{variable}_max"] = serie[f"{variable}_estimado"] / (1 + bajo)
    serie.index = a_fecha(serie.index)
    return serie


serie = serie_armonizada(celdas_armonizadas)
serie.to_csv(ROOT / "data" / "modelo" / "serie_armonizada.csv", index_label="mes")
graf.serie(serie)

---
## 12. Before and after harmonizing
The chart sums up what all this is for. If the records are joined as they are, MODIS until 2011 and VIIRS from 2012, a false jump appears in the detections in 2012: VIIRS sees about 4.6 times more, but there was no more fire, only the sensor changed. Harmonized, the record is continuous.

In total FRP there is almost no jump (1.05×). Both sensors measure a similar total energy; VIIRS just spreads it over more pixels, which are smaller.

Each point is the monthly average of one year. Only years with at least 9 good months are used.

In [ ]:
buenos = serie.notna().groupby(serie.index.year).sum()
anual = serie.groupby(serie.index.year).mean(numeric_only=True)
for columna in anual.columns:
    anual.loc[buenos[columna] < 9, columna] = np.nan

coinciden = anual.loc[2012:2021]  # years with both MODIS and VIIRS
factor = {v: (coinciden[f"{v}_observado"] / coinciden[f"{v}_modis"]).mean() for v in ("detecciones", "frp")}
graf.antes_y_despues(anual, factor)